In [ ]:
#!/usr/bin/env python3
"""
Эксперименты с планированием траекторий на основе принципа максимума Понтрягина (ПМП)
для манипулятора UR5e.

Эксперименты:
  1. Профили траектории ПМП minimum-jerk (q, dq, ddq, jerk, костаты, стоимость)
  2. Влияние длительности T на стоимость jerk
  3. Ненулевые граничные условия (начальные/конечные скорости)
  4. Многоточечные траектории (waypoints)
  5. Сравнение ПМП с трапецеидальным профилем скорости
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os

OUTPUT_DIR = "pmp_graphs_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# Реализация ПМП minimum-jerk (Python, эквивалент trajectory.hpp)
# ============================================================

def quintic_coeffs(q0, v0, a0, q1, v1, a1, T):
    """Коэффициенты полинома 5-й степени с общими граничными условиями."""
    A = np.zeros((6, 6))
    b = np.zeros(6)
    A[0, 0] = 1.0; b[0] = q0
    A[1, 1] = 1.0; b[1] = v0
    A[2, 2] = 2.0; b[2] = a0
    A[3] = [1, T, T**2, T**3, T**4, T**5]; b[3] = q1
    A[4] = [0, 1, 2*T, 3*T**2, 4*T**3, 5*T**4]; b[4] = v1
    A[5] = [0, 0, 2, 6*T, 12*T**2, 20*T**3]; b[5] = a1
    return np.linalg.solve(A, b)


def plan_pmp_minimum_jerk(q0, dq0, ddq0, q1, dq1, ddq1, T, dt=0.001):
    """
    Планирование траектории minimum-jerk с помощью ПМП.
    Возвращает словарь с t, q, dq, ddq, u (jerk), lambda1-3, J_acc.
    """
    dof = len(q0)
    N = max(2, int(np.ceil(T / dt)))
    dt_eff = T / N

    coeffs = []
    for i in range(dof):
        coeffs.append(quintic_coeffs(q0[i], dq0[i], ddq0[i], q1[i], dq1[i], ddq1[i], T))

    t_arr = np.linspace(0, T, N + 1)
    q_arr = np.zeros((N + 1, dof))
    dq_arr = np.zeros((N + 1, dof))
    ddq_arr = np.zeros((N + 1, dof))
    u_arr = np.zeros((N + 1, dof))
    lam1 = np.zeros((N + 1, dof))
    lam2 = np.zeros((N + 1, dof))
    lam3 = np.zeros((N + 1, dof))
    J_acc = np.zeros(N + 1)

    cost = 0.0
    for k in range(N + 1):
        t = t_arr[k]
        tp = np.array([1, t, t**2, t**3, t**4, t**5])
        tp_d1 = np.array([0, 1, 2*t, 3*t**2, 4*t**3, 5*t**4])
        tp_d2 = np.array([0, 0, 2, 6*t, 12*t**2, 20*t**3])
        tp_d3 = np.array([0, 0, 0, 6, 24*t, 60*t**2])

        for i in range(dof):
            a = coeffs[i]
            q_arr[k, i] = a @ tp
            dq_arr[k, i] = a @ tp_d1
            ddq_arr[k, i] = a @ tp_d2
            u_arr[k, i] = a @ tp_d3
            lam3[k, i] = -u_arr[k, i]
            du_dt = 24*a[4] + 120*a[5]*t
            d2u_dt2 = 120*a[5]
            lam2[k, i] = du_dt
            lam1[k, i] = -d2u_dt2

        u2 = np.sum(u_arr[k]**2)
        if k > 0:
            cost += 0.5 * u2 * dt_eff
        J_acc[k] = cost

    return {
        't': t_arr, 'q': q_arr, 'dq': dq_arr, 'ddq': ddq_arr,
        'u': u_arr, 'lambda1': lam1, 'lambda2': lam2, 'lambda3': lam3,
        'J_acc': J_acc, 'J_total': cost
    }


def trapezoidal_velocity_profile(q0, q1, T, dt=0.001, accel_fraction=0.3):
    """Трапецеидальный профиль скорости для сравнения."""
    dof = len(q0)
    N = max(2, int(np.ceil(T / dt)))
    t_arr = np.linspace(0, T, N + 1)
    q_arr = np.zeros((N + 1, dof))
    dq_arr = np.zeros((N + 1, dof))
    ddq_arr = np.zeros((N + 1, dof))
    u_arr = np.zeros((N + 1, dof))

    ta = accel_fraction * T
    tc = T - 2 * ta

    for i in range(dof):
        delta = q1[i] - q0[i]
        v_max = delta / (T - ta)
        a_max = v_max / ta

        for k, t in enumerate(t_arr):
            if t <= ta:
                q_arr[k, i] = q0[i] + 0.5 * a_max * t**2
                dq_arr[k, i] = a_max * t
                ddq_arr[k, i] = a_max
            elif t <= ta + tc:
                q_arr[k, i] = q0[i] + 0.5 * a_max * ta**2 + v_max * (t - ta)
                dq_arr[k, i] = v_max
                ddq_arr[k, i] = 0.0
            else:
                t_dec = t - ta - tc
                q_arr[k, i] = q0[i] + 0.5 * a_max * ta**2 + v_max * tc + v_max * t_dec - 0.5 * a_max * t_dec**2
                dq_arr[k, i] = v_max - a_max * t_dec
                ddq_arr[k, i] = -a_max

    for k in range(1, N + 1):
        u_arr[k] = (ddq_arr[k] - ddq_arr[k-1]) / (t_arr[k] - t_arr[k-1])

    J_acc = np.zeros(N + 1)
    cost = 0.0
    dt_eff = T / N
    for k in range(1, N + 1):
        cost += 0.5 * np.sum(u_arr[k]**2) * dt_eff
        J_acc[k] = cost

    return {
        't': t_arr, 'q': q_arr, 'dq': dq_arr, 'ddq': ddq_arr,
        'u': u_arr, 'J_acc': J_acc, 'J_total': cost
    }


# ============================================================
# Конфигурации UR5e
# ============================================================

UR5E_JOINT_NAMES = ['Основание', 'Плечо', 'Локоть', 'Запястье 1', 'Запястье 2', 'Запястье 3']
UR5E_TAU_MAX = [150.0, 150.0, 150.0, 28.0, 28.0, 28.0]

Q_HOME = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
Q_TARGET1 = np.array([np.pi/4, -np.pi/3, np.pi/6, -np.pi/4, np.pi/3, np.pi/6])
Q_TARGET2 = np.array([-np.pi/6, -np.pi/4, np.pi/4, -np.pi/3, -np.pi/6, np.pi/4])

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'lines.linewidth': 1.5,
    'figure.facecolor': 'white',
})

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# ============================================================
# Эксперимент 1: Профили траектории ПМП
# ============================================================
print("=== Эксперимент 1: Профили траектории ПМП ===")

q0 = Q_HOME
q1 = Q_TARGET1
T = 3.0
zeros6 = np.zeros(6)
traj = plan_pmp_minimum_jerk(q0, zeros6, zeros6, q1, zeros6, zeros6, T, dt=0.001)

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('Эксперимент 1: Профили траектории ПМП Minimum-Jerk\n(UR5e, T=3с, покой\u2192покой)', fontsize=14, fontweight='bold')

ax = axes[0, 0]
for i in range(6):
    ax.plot(traj['t'], np.degrees(traj['q'][:, i]), color=COLORS[i], label=UR5E_JOINT_NAMES[i])
ax.set_ylabel('Положение (\u00b0)')
ax.set_title('а) Положение суставов q(t)')
ax.legend(loc='best', ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for i in range(6):
    ax.plot(traj['t'], np.degrees(traj['dq'][:, i]), color=COLORS[i], label=UR5E_JOINT_NAMES[i])
ax.set_ylabel('Скорость (\u00b0/с)')
ax.set_title('б) Скорость суставов dq(t)')
ax.legend(loc='best', ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for i in range(6):
    ax.plot(traj['t'], np.degrees(traj['ddq'][:, i]), color=COLORS[i], label=UR5E_JOINT_NAMES[i])
ax.set_ylabel('Ускорение (\u00b0/с\u00b2)')
ax.set_title('в) Ускорение суставов ddq(t)')
ax.legend(loc='best', ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
for i in range(6):
    ax.plot(traj['t'], traj['u'][:, i], color=COLORS[i], label=UR5E_JOINT_NAMES[i])
ax.set_ylabel('Jerk (рад/с\u00b3)')
ax.set_title('г) Оптимальное управление u*(t) = jerk')
ax.legend(loc='best', ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[2, 0]
for i in range(6):
    ax.plot(traj['t'], traj['lambda3'][:, i], color=COLORS[i], label=f'\u03bb\u2083 {UR5E_JOINT_NAMES[i]}')
ax.set_ylabel('\u03bb\u2083(t)')
ax.set_xlabel('Время (с)')
ax.set_title('д) Сопряжённая переменная \u03bb\u2083(t) = \u2212u*(t)')
ax.legend(loc='best', ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[2, 1]
ax.plot(traj['t'], traj['J_acc'], 'k-', linewidth=2)
ax.set_ylabel('J(t)')
ax.set_xlabel('Время (с)')
ax.set_title(f'е) Накопленная стоимость J = \u222b\u00bd||u||\u00b2dt = {traj["J_total"]:.4f}')
ax.grid(True, alpha=0.3)
ax.fill_between(traj['t'], 0, traj['J_acc'], alpha=0.15, color='steelblue')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_pmp_profiles.png'), dpi=150, bbox_inches='tight')
plt.close()
print(f"  Суммарная стоимость J = {traj['J_total']:.6f}")

# ============================================================
# Эксперимент 2: Влияние длительности T на стоимость
# ============================================================
print("\n=== Эксперимент 2: Влияние T на стоимость jerk ===")

T_values = np.linspace(0.5, 8.0, 30)
J_values = []
max_jerk_values = []
max_vel_values = []
max_accel_values = []

for T_val in T_values:
    res = plan_pmp_minimum_jerk(q0, zeros6, zeros6, q1, zeros6, zeros6, T_val, dt=0.002)
    J_values.append(res['J_total'])
    max_jerk_values.append(np.max(np.abs(res['u'])))
    max_vel_values.append(np.max(np.abs(res['dq'])))
    max_accel_values.append(np.max(np.abs(res['ddq'])))

J_values = np.array(J_values)
max_jerk_values = np.array(max_jerk_values)
max_vel_values = np.array(max_vel_values)
max_accel_values = np.array(max_accel_values)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Эксперимент 2: Влияние длительности T на траекторию ПМП', fontsize=14, fontweight='bold')

ax = axes[0, 0]
ax.plot(T_values, J_values, 'b-o', markersize=4)
ax.set_ylabel('Стоимость J')
ax.set_xlabel('Длительность T (с)')
ax.set_title('а) Функционал стоимости J от T')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(T_values, max_jerk_values, 'r-o', markersize=4)
ax.set_ylabel('|u|_max (рад/с\u00b3)')
ax.set_xlabel('Длительность T (с)')
ax.set_title('б) Максимальный jerk от T')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(T_values, np.degrees(max_vel_values), 'g-o', markersize=4)
ax.set_ylabel('|dq|_max (\u00b0/с)')
ax.set_xlabel('Длительность T (с)')
ax.set_title('в) Максимальная скорость от T')
ax.axhline(y=np.degrees(np.pi), color='r', linestyle='--', alpha=0.5, label='Предел суставов 1\u20133')
ax.axhline(y=np.degrees(2*np.pi), color='orange', linestyle='--', alpha=0.5, label='Предел суставов 4\u20136')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(T_values, np.degrees(max_accel_values), 'm-o', markersize=4)
ax.set_ylabel('|ddq|_max (\u00b0/с\u00b2)')
ax.set_xlabel('Длительность T (с)')
ax.set_title('г) Максимальное ускорение от T')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_effect_of_T.png'), dpi=150, bbox_inches='tight')
plt.close()

hdr_dq = '|dq|_max (°/с)'
hdr_ddq = '|ddq|_max (°/с²)'
print(f"  {'T (с)':>6}  {'J':>12}  {'|u|_max':>10}  {hdr_dq:>15}  {hdr_ddq:>18}")
for i in range(0, len(T_values), 5):
    print(f"  {T_values[i]:6.1f}  {J_values[i]:12.4f}  {max_jerk_values[i]:10.4f}  {np.degrees(max_vel_values[i]):15.2f}  {np.degrees(max_accel_values[i]):18.2f}")

# ============================================================
# Эксперимент 3: Ненулевые граничные условия
# ============================================================
print("\n=== Эксперимент 3: Ненулевые граничные условия ===")

T = 3.0
conditions = [
    ("Покой \u2192 Покой", zeros6, zeros6, zeros6, zeros6),
    ("v\u2080=0.5 \u2192 Покой", np.full(6, 0.5), zeros6, zeros6, zeros6),
    ("Покой \u2192 v\u2081=0.5", zeros6, zeros6, np.full(6, 0.5), zeros6),
    ("v\u2080=0.5 \u2192 v\u2081=\u22120.5", np.full(6, 0.5), zeros6, np.full(6, -0.5), zeros6),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Эксперимент 3: Влияние граничных условий на ПМП\n(Сустав 1 \u2014 Основание, T=3с)', fontsize=14, fontweight='bold')

joint_idx = 0

for idx, (name, dq0_bc, ddq0_bc, dq1_bc, ddq1_bc) in enumerate(conditions):
    res = plan_pmp_minimum_jerk(q0, dq0_bc, ddq0_bc, q1, dq1_bc, ddq1_bc, T)
    axes[0, 0].plot(res['t'], np.degrees(res['q'][:, joint_idx]), color=COLORS[idx], label=name)
    axes[0, 1].plot(res['t'], np.degrees(res['dq'][:, joint_idx]), color=COLORS[idx], label=name)
    axes[1, 0].plot(res['t'], np.degrees(res['ddq'][:, joint_idx]), color=COLORS[idx], label=name)
    axes[1, 1].plot(res['t'], res['u'][:, joint_idx], color=COLORS[idx], label=name)
    print(f"  {name:25s}  J = {res['J_total']:.6f}")

axes[0, 0].set_title('а) Положение q\u2081(t)'); axes[0, 0].set_ylabel('Положение (\u00b0)')
axes[0, 1].set_title('б) Скорость dq\u2081(t)'); axes[0, 1].set_ylabel('Скорость (\u00b0/с)')
axes[1, 0].set_title('в) Ускорение ddq\u2081(t)'); axes[1, 0].set_ylabel('Ускорение (\u00b0/с\u00b2)')
axes[1, 1].set_title('г) Управление u\u2081*(t) = jerk'); axes[1, 1].set_ylabel('Jerk (рад/с\u00b3)')

for ax in axes.flat:
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('Время (с)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_boundary_conditions.png'), dpi=150, bbox_inches='tight')
plt.close()

# ============================================================
# Эксперимент 4: Многоточечные траектории (waypoints)
# ============================================================
print("\n=== Эксперимент 4: Многоточечные траектории ===")

waypoints = [Q_HOME, Q_TARGET1, Q_TARGET2, Q_HOME]
T_segments = [2.0, 2.0, 2.0]

t_full = []
q_full = []
dq_full = []
ddq_full = []
u_full = []
J_segments = []
t_offset = 0.0

for seg in range(len(T_segments)):
    T_seg = T_segments[seg]
    q_start = waypoints[seg]
    q_end = waypoints[seg + 1]
    res = plan_pmp_minimum_jerk(q_start, zeros6, zeros6, q_end, zeros6, zeros6, T_seg, dt=0.002)
    start_idx = 1 if seg > 0 else 0
    t_full.extend(res['t'][start_idx:] + t_offset)
    q_full.extend(res['q'][start_idx:])
    dq_full.extend(res['dq'][start_idx:])
    ddq_full.extend(res['ddq'][start_idx:])
    u_full.extend(res['u'][start_idx:])
    J_segments.append(res['J_total'])
    t_offset += T_seg

t_full = np.array(t_full)
q_full = np.array(q_full)
dq_full = np.array(dq_full)
u_full = np.array(u_full)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Эксперимент 4: Многоточечная траектория ПМП\n(3 сегмента: Дом\u2192Цель1\u2192Цель2\u2192Дом)', fontsize=14, fontweight='bold')

wp_times = [0] + list(np.cumsum(T_segments))

ax = axes[0, 0]
for i in range(6):
    ax.plot(t_full, np.degrees(q_full[:, i]), color=COLORS[i], label=UR5E_JOINT_NAMES[i])
for wt in wp_times[1:-1]:
    ax.axvline(x=wt, color='gray', linestyle='--', alpha=0.5)
ax.set_title('а) Положение суставов q(t)')
ax.set_ylabel('Положение (\u00b0)')
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for i in range(6):
    ax.plot(t_full, np.degrees(dq_full[:, i]), color=COLORS[i])
for wt in wp_times[1:-1]:
    ax.axvline(x=wt, color='gray', linestyle='--', alpha=0.5)
ax.set_title('б) Скорость суставов dq(t)')
ax.set_ylabel('Скорость (\u00b0/с)')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for i in range(6):
    ax.plot(t_full, u_full[:, i], color=COLORS[i])
for wt in wp_times[1:-1]:
    ax.axvline(x=wt, color='gray', linestyle='--', alpha=0.5)
ax.set_title('в) Оптимальное управление u*(t) = jerk')
ax.set_ylabel('Jerk (рад/с\u00b3)')
ax.set_xlabel('Время (с)')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.bar(['Сег. 1\nДом\u2192Ц1', 'Сег. 2\nЦ1\u2192Ц2', 'Сег. 3\nЦ2\u2192Дом'], J_segments, color=['steelblue', 'coral', 'seagreen'])
ax.set_title(f'г) Стоимость по сегментам (Итого: {sum(J_segments):.4f})')
ax.set_ylabel('Стоимость J')
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(J_segments):
    ax.text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_multi_waypoint.png'), dpi=150, bbox_inches='tight')
plt.close()

print(f"  Стоимости по сегментам: {[f'{j:.4f}' for j in J_segments]}")
print(f"  Общая стоимость: {sum(J_segments):.4f}")

# ============================================================
# Эксперимент 5: Сравнение ПМП с трапецеидальным профилем
# ============================================================
print("\n=== Эксперимент 5: ПМП vs Трапецеидальный профиль ===")

T = 3.0
pmp_res = plan_pmp_minimum_jerk(q0, zeros6, zeros6, q1, zeros6, zeros6, T)
trap_res = trapezoidal_velocity_profile(q0, q1, T, dt=0.001)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Эксперимент 5: ПМП Minimum-Jerk vs Трапецеидальный профиль\n(Сустав 1 \u2014 Основание, T=3с)', fontsize=14, fontweight='bold')

ji = 0

ax = axes[0, 0]
ax.plot(pmp_res['t'], np.degrees(pmp_res['q'][:, ji]), 'b-', label='ПМП', linewidth=2)
ax.plot(trap_res['t'], np.degrees(trap_res['q'][:, ji]), 'r--', label='Трапецеид.', linewidth=2)
ax.set_title('а) Положение q\u2081(t)')
ax.set_ylabel('Положение (\u00b0)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(pmp_res['t'], np.degrees(pmp_res['dq'][:, ji]), 'b-', label='ПМП', linewidth=2)
ax.plot(trap_res['t'], np.degrees(trap_res['dq'][:, ji]), 'r--', label='Трапецеид.', linewidth=2)
ax.set_title('б) Скорость dq\u2081(t)')
ax.set_ylabel('Скорость (\u00b0/с)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(pmp_res['t'], np.degrees(pmp_res['ddq'][:, ji]), 'b-', label='ПМП', linewidth=2)
ax.plot(trap_res['t'], np.degrees(trap_res['ddq'][:, ji]), 'r--', label='Трапецеид.', linewidth=2)
ax.set_title('в) Ускорение ddq\u2081(t)')
ax.set_ylabel('Ускорение (\u00b0/с\u00b2)')
ax.set_xlabel('Время (с)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(pmp_res['t'], pmp_res['J_acc'], 'b-', label=f'ПМП (J={pmp_res["J_total"]:.4f})', linewidth=2)
ax.plot(trap_res['t'], trap_res['J_acc'], 'r--', label=f'Трапецеид. (J={trap_res["J_total"]:.4f})', linewidth=2)
ax.set_title('г) Накопленная стоимость J(t)')
ax.set_ylabel('J(t)')
ax.set_xlabel('Время (с)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_pmp_vs_trapezoidal.png'), dpi=150, bbox_inches='tight')
plt.close()

ratio = trap_res['J_total'] / pmp_res['J_total']
print(f"  ПМП J = {pmp_res['J_total']:.6f}")
print(f"  Трапецеид. J = {trap_res['J_total']:.6f}")
print(f"  Соотношение (Трап/ПМП) = {ratio:.2f}x")

# ============================================================
# Сводная таблица метрик
# ============================================================
print("\n=== Сводная таблица ===")

summary_T = [1.0, 2.0, 3.0, 4.0, 5.0]
summary_data = []
for Tv in summary_T:
    r_pmp = plan_pmp_minimum_jerk(q0, zeros6, zeros6, q1, zeros6, zeros6, Tv)
    r_trap = trapezoidal_velocity_profile(q0, q1, Tv)
    summary_data.append({
        'T': Tv,
        'J_pmp': r_pmp['J_total'],
        'J_trap': r_trap['J_total'],
        'ratio': r_trap['J_total'] / max(r_pmp['J_total'], 1e-12),
        'max_vel_pmp': np.degrees(np.max(np.abs(r_pmp['dq']))),
        'max_jerk_pmp': np.max(np.abs(r_pmp['u'])),
    })

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')
table_data = [['T (с)', 'J_ПМП', 'J_Трап', 'Соотн.', 'Скор_макс ПМП (\u00b0/с)', 'Jerk_макс ПМП']]
for d in summary_data:
    table_data.append([
        f"{d['T']:.1f}",
        f"{d['J_pmp']:.4f}",
        f"{d['J_trap']:.4f}",
        f"{d['ratio']:.2f}x",
        f"{d['max_vel_pmp']:.2f}",
        f"{d['max_jerk_pmp']:.4f}",
    ])

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

for j in range(len(table_data[0])):
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')

for i in range(1, len(table_data)):
    color = '#D6E4F0' if i % 2 == 0 else 'white'
    for j in range(len(table_data[0])):
        table[i, j].set_facecolor(color)

plt.title('Сводная таблица: ПМП vs Трапецеидальный профиль', fontsize=13, fontweight='bold', pad=20)
plt.savefig(os.path.join(OUTPUT_DIR, '06_summary_table.png'), dpi=150, bbox_inches='tight')
plt.close()

print("\n\u2713 Все эксперименты завершены. Графики сохранены в:", OUTPUT_DIR)


NameError: name '__file__' is not defined